# Bus travel-time baseline: separate versus pooled models
**Joshua Ribeiro — DS 340W**

Upload the filtered MTA segment-speed CSV and run cells in order (Colab: Runtime → Run all). A CPU runtime is sufficient. The supplied dataset has 189,933 rows, January 2025–August 2026. We keep all hours and both directions; no extra downloads are needed.

**Question:** Does combining M101/M102/M103 improve prediction of monthly weekday/hour average segment travel times? This is aggregate forecasting, not individual bus delay prediction or causal propagation.

This notebook cleans the data, exports an audit, creates historical features, tunes modest gradient-boosted tree models on earlier months, evaluates untouched later months, and exports predictions, metrics, plots and fitted models. Scikit-learn's HistGradientBoostingRegressor is used, **not XGBoost**. Historical mean and historical-lag predictors provide strong simple benchmarks.

**Timing assumption:** use observations at least two calendar months old. Actual dataset publication dates are not supplied: this is a retrospective experiment under an explicit two-month availability assumption, not a verified live forecasting system. Increase `LAG_MONTHS` if operational publication delays require it. Test-month historical features may use earlier test outcomes once allowed by that lag; model parameters remain fixed.

**Cleaning choice:** repeated endpoint/time keys with differing source values are quarantined, not silently averaged. They may represent unrecorded variants. This makes the first experiment conservative and limits conclusions to unambiguous groups. Raw data is never overwritten.


## Verified run on the supplied CSV
The complete notebook ran successfully. After quarantining 23,157 ambiguous rows, 166,776 rows remain. Final evaluation uses 25,534 history-eligible records in June–August 2026.

| Method | MAE (minutes) |
|---|---:|
| Historical mean | 0.7114 |
| Historical lag | 0.8395 |
| Separate boosting | 0.6860 |
| Pooled boosting | 0.6903 |

Pooling was **0.62% worse** than separate models on aggregate MAE in this run; the difference is small and has not been tested for statistical significance. Separate boosting improved over historical mean by about 3.57%. These results support a runnable baseline, not a claim that pooling helps. Inspect route-level and monthly results before drawing broader conclusions.

**To rerun:** upload this notebook to Colab, choose Runtime → Run all, then upload the original CSV when prompted. CPU is sufficient. The final cell writes `bus_baseline_results.zip`; download it through Colab’s Files panel or set `DOWNLOAD_RESULTS = True`. Stored outputs here describe the supplied CSV and may differ after changing inputs or software versions.


In [ ]:
from pathlib import Path
import os, json, hashlib, platform, warnings, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
from IPython.display import display

SEED = 42
ROUTES = ['M101', 'M102', 'M103']
LAG_MONTHS = 2
OUT = Path('bus_baseline_outputs')
OUT.mkdir(exist_ok=True)
# For local use, set BUS_CSV to your file path. In Colab, the upload dialog opens.
CSV_PATH = os.environ.get('BUS_CSV', '')
if not CSV_PATH:
    from google.colab import files
    uploaded = files.upload()
    choices = [name for name in uploaded if name.lower().endswith('.csv')]
    if len(choices) != 1:
        raise ValueError('Please upload exactly one source CSV.')
    CSV_PATH = choices[0]
print('Source:', Path(CSV_PATH).name)
print('Versions:', {'python': platform.python_version(), 'pandas': pd.__version__,
                    'sklearn': sklearn.__version__})


## 1. Clean and audit
IDs are categorical labels. Invalid mandatory fields are excluded; slow but positive travel times are retained. Low trip counts are flagged, not automatically removed. Exact duplicates are removed first; all remaining ambiguous key groups are quarantined. No imputation of the target is performed.

In [ ]:
raw = pd.read_csv(CSV_PATH, dtype=str)
raw.columns = raw.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)
required = ['year','month','day_of_week','hour_of_day','route_id','direction','route_type',
            'timepoint_stop_id','next_timepoint_stop_id','road_distance',
            'average_travel_time','bus_trip_count']
missing = sorted(set(required) - set(raw.columns))
if missing:
    raise ValueError(f'Missing required columns: {missing}')
raw['source_row'] = np.arange(len(raw)) + 2
for col in required:
    raw[col] = raw[col].str.strip()
raw['route_id'] = raw.route_id.str.upper()
d = raw[raw.route_id.isin(ROUTES)].copy()
audit = {'source_rows': len(raw), 'selected_route_rows': len(d)}
source_cols = [c for c in d.columns if c != 'source_row']
exact = d.duplicated(source_cols)
audit['exact_duplicates_removed'] = int(exact.sum())
d.loc[exact].to_csv(OUT/'exact_duplicates.csv', index=False)
d = d.loc[~exact].copy()
nums = ['year','month','hour_of_day','road_distance','average_travel_time','bus_trip_count']
for col in nums:
    d[col] = pd.to_numeric(d[col], errors='coerce')
for col in ['timepoint_stop_id','next_timepoint_stop_id']:
    d[col] = d[col].str.replace(r'\.0$', '', regex=True)
d['day_of_week'] = d.day_of_week.str.title()
d['direction'] = d.direction.str.upper()
weekdays = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
valid = d[required].notna().all(axis=1) & np.isfinite(d[nums]).all(axis=1)
valid &= d.month.between(1,12) & d.hour_of_day.between(0,23)
valid &= d.year.between(2000,2100)
for col in ['year','month','hour_of_day','bus_trip_count']:
    valid &= d[col].eq(d[col].round())
valid &= d.day_of_week.isin(weekdays) & d.direction.isin(['N','S','E','W'])
valid &= (d[['road_distance','average_travel_time','bus_trip_count']] > 0).all(axis=1)
valid &= d.timepoint_stop_id.ne(d.next_timepoint_stop_id)
for col in ['route_type','timepoint_stop_id','next_timepoint_stop_id']:
    valid &= d[col].ne('')
d.loc[~valid].to_csv(OUT/'invalid_rows.csv', index=False)
audit['invalid_rows_removed'] = int((~valid).sum())
d = d.loc[valid].copy()
for col in ['year','month','hour_of_day','bus_trip_count']:
    d[col] = d[col].astype(int)
series_key = ['route_id','direction','route_type','timepoint_stop_id',
              'next_timepoint_stop_id','day_of_week','hour_of_day']
key = ['year','month'] + series_key
ambiguous = d.duplicated(key, keep=False)
d.loc[ambiguous].to_csv(OUT/'ambiguous_groups.csv', index=False)
audit['ambiguous_rows_quarantined'] = int(ambiguous.sum())
audit['ambiguous_groups'] = len(d.loc[ambiguous,key].drop_duplicates())
d = d.loc[~ambiguous].copy()
d['period'] = pd.PeriodIndex(year=d.year, month=d.month, freq='M')
d['month_index'] = d.year*12 + d.month
# No same-period observed speed, trip count or distance is passed as a model predictor.
d['low_support'] = d.bus_trip_count < 5
d['segment_id'] = d.timepoint_stop_id + '>' + d.next_timepoint_stop_id + ':' + d.direction
assert not d.duplicated(key).any()
audit['clean_rows'] = len(d)
audit['low_support_rows_retained'] = int(d.low_support.sum())
d.sort_values(key).to_csv(OUT/'cleaned_data.csv', index=False)
display(pd.Series(audit, name='count').to_frame())
coverage = d.groupby(['period','route_id']).size().unstack(fill_value=0)
coverage.to_csv(OUT/'coverage.csv')
display(coverage)
display(d.groupby('route_id')[['average_travel_time','bus_trip_count']].agg(['min','median','max']))
segment_coverage = d.groupby(['period','route_id']).segment_id.nunique().unstack(fill_value=0)
segment_coverage.to_csv(OUT/'segment_coverage.csv')
shared = d[['segment_id','route_id']].drop_duplicates().groupby('segment_id').route_id.nunique()
print('Endpoint/direction pairs present on multiple routes:', int((shared > 1).sum()))
print('These are candidate shared segments, not verified identical road paths.')


## 2. Create features from earlier months only
The series key includes route, service type, direction, endpoints, weekday and hour. Lags are exact calendar-month matches, never the previous available row. Missing lags remain missing and are imputed using training data only. Distance also comes from earlier observations. Same-period average speed, trip count, target and source timestamp are excluded from features.

Historical mean = mean of all eligible past monthly averages for the same key (not weighted by trips). Lag baseline falls back to historical mean, then a training-only route mean when a series has no history.

In [ ]:
data = d.copy()
for lag in [LAG_MONTHS, LAG_MONTHS+1, LAG_MONTHS+2]:
    hist = d[series_key + ['month_index','average_travel_time']].copy()
    hist['month_index'] += lag
    hist = hist.rename(columns={'average_travel_time':f'lag_{lag}'})
    data = data.merge(hist, on=series_key+['month_index'], how='left', validate='one_to_one')
# Shift past expanding statistics forward to the earliest permitted target month.
parts = []
for _, g in d.groupby(series_key, sort=False):
    g = g.sort_values('month_index').copy()
    g['history_mean'] = g.average_travel_time.expanding().mean()
    g['historical_distance'] = g.road_distance.expanding().mean()
    g['history_count'] = np.arange(1,len(g)+1)
    g['available_month'] = g.month_index + LAG_MONTHS
    parts.append(g[series_key+['available_month','history_mean','historical_distance','history_count']])
h = pd.concat(parts, ignore_index=True)
data = pd.merge_asof(data.sort_values('month_index'), h.sort_values('available_month'),
                     left_on='month_index', right_on='available_month', by=series_key,
                     direction='backward')
assert (data.available_month.dropna() <= data.loc[data.available_month.notna(),'month_index']).all()
data['month_sin'] = np.sin(2*np.pi*data.month/12)
data['month_cos'] = np.cos(2*np.pi*data.month/12)
data = data.sort_values(['month_index']+series_key).reset_index(drop=True)
# First months cannot supply supervised examples with any eligible history.
eligible = data.history_count.notna()
audit['rows_without_eligible_history'] = int((~eligible).sum())
data = data.loc[eligible].copy()
months = sorted(data.month_index.unique())
if len(months) < 10:
    raise ValueError('Need at least 10 usable target months for these held-out splits.')
test_months = months[-3:]
first_test = min(test_months)
refit_end = first_test-LAG_MONTHS
val_months = [refit_end-2, refit_end-1]
train_end = min(val_months)-LAG_MONTHS
train = data[data.month_index <= train_end].copy()
val = data[data.month_index.isin(val_months)].copy()
refit = data[data.month_index <= refit_end].copy()
test = data[data.month_index.isin(test_months)].copy()
for name, frame in [('train',train),('validation',val),('refit',refit),('test',test)]:
    assert len(frame) and set(ROUTES).issubset(frame.route_id.unique()), name
    print(name, str(frame.period.min()), 'to', str(frame.period.max()), 'rows:', len(frame))
assert train.month_index.max() <= val.month_index.min()-LAG_MONTHS
assert refit.month_index.max() <= test.month_index.min()-LAG_MONTHS
print('Test rows with exact lag:', round(test[f'lag_{LAG_MONTHS}'].notna().mean()*100,2),'%')
data.to_csv(OUT/'modeling_data.csv', index=False)


## 3. Tune a small gradient-boosting comparison
Two deliberately modest configurations are selected on validation MAE only. Separate models may choose different configurations per route; the pooled model has the same two-configuration search budget. All models use the same features and target rows. HistGradientBoosting fits residual patterns using trees; it is a different implementation from XGBoost. Early stopping is disabled to avoid its random internal validation split.

Final models are refitted through the allowed cutoff, frozen, and evaluated on the three later months. No test results are used for tuning.

In [ ]:
categorical = ['route_id','direction','route_type','segment_id','day_of_week']
numeric = ['hour_of_day','month_sin','month_cos','month_index','historical_distance',
           'history_mean','history_count'] + [f'lag_{k}' for k in [LAG_MONTHS,LAG_MONTHS+1,LAG_MONTHS+2]]
features = categorical + numeric
assert not set(features) & {'average_travel_time','average_road_speed','bus_trip_count','road_distance'}
configs = [dict(max_leaf_nodes=15,l2_regularization=10.0),
           dict(max_leaf_nodes=31,l2_regularization=10.0)]
def make_model(config):
    prep = ColumnTransformer([
        ('category', OneHotEncoder(handle_unknown='ignore',sparse_output=False),categorical),
        ('number', SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True),numeric)
    ])
    model = HistGradientBoostingRegressor(learning_rate=0.08,max_iter=150,
             min_samples_leaf=30,early_stopping=False,random_state=SEED,**config)
    return Pipeline([('prepare',prep),('model',model)])
models, selections, validation_rows = {}, {}, []
for scope in ['pooled'] + ROUTES:
    tr = train if scope == 'pooled' else train[train.route_id.eq(scope)]
    va = val if scope == 'pooled' else val[val.route_id.eq(scope)]
    scores = []
    for i, config in enumerate(configs):
        model = make_model(config)
        model.fit(tr[features],tr.average_travel_time)
        pred = np.maximum(0,model.predict(va[features]))
        score = mean_absolute_error(va.average_travel_time,pred)
        scores.append(score)
        validation_rows.append({'scope':scope,'configuration':i,'validation_MAE':score})
    best = int(np.argmin(scores))
    selections[scope] = best
    fit = refit if scope == 'pooled' else refit[refit.route_id.eq(scope)]
    models[scope] = make_model(configs[best]).fit(fit[features],fit.average_travel_time)
    print(scope,'selected configuration',best,'validation MAE',round(scores[best],3))
validation_results = pd.DataFrame(validation_rows)
validation_results.to_csv(OUT/'validation_results.csv',index=False)


## 4. Score all approaches on identical held-out rows
MAE and RMSE are in minutes; smaller is better. Trip-weighted MAE is secondary: it weights errors in aggregate means by their sample counts, not errors on individual bus trips. Improvements are descriptive; three test months do not establish statistical significance or generalization to every season. Low-support and shared-segment diagnostics are exported without changing the test set.

In [ ]:
p = test.copy()
route_means = refit.groupby('route_id').average_travel_time.mean()
fallback = p.route_id.map(route_means).fillna(refit.average_travel_time.mean())
p['Historical mean'] = p.history_mean.fillna(fallback)
p['Historical lag'] = p[f'lag_{LAG_MONTHS}'].fillna(p['Historical mean'])
p['Pooled boosting'] = np.maximum(0,models['pooled'].predict(p[features]))
p['Separate boosting'] = np.nan
for route in ROUTES:
    mask = p.route_id.eq(route)
    p.loc[mask,'Separate boosting'] = np.maximum(0,models[route].predict(p.loc[mask,features]))
names = ['Historical mean','Historical lag','Separate boosting','Pooled boosting']
assert p[names].notna().all().all()
def score_table(frame, group):
    rows = []
    groups = [('ALL',frame)] if group is None else frame.groupby(group,observed=True)
    for label,g in groups:
        y = g.average_travel_time.to_numpy()
        for name in names:
            err = y-g[name].to_numpy()
            rows.append({'group':str(label),'model':name,'rows':len(g),
              'MAE_minutes':np.mean(np.abs(err)), 'RMSE_minutes':np.sqrt(np.mean(err**2)),
              'trip_weighted_MAE':np.average(np.abs(err),weights=g.bus_trip_count)})
    return pd.DataFrame(rows)
metrics = score_table(p,None)
by_route = score_table(p,'route_id')
by_month = score_table(p,'period')
p['candidate_shared_segment'] = p.segment_id.map(shared).fillna(0).gt(1)
for name,table in [('metrics',metrics),('metrics_by_route',by_route),('metrics_by_month',by_month),
                   ('metrics_by_support',score_table(p,'low_support')),
                   ('metrics_by_shared_segment',score_table(p,'candidate_shared_segment'))]:
    table.to_csv(OUT/f'{name}.csv',index=False)
display(metrics.round(4)); display(by_route.round(4)); display(by_month.round(4))
maes = metrics.set_index('model').MAE_minutes
improvement = 100*(maes['Separate boosting']-maes['Pooled boosting'])/maes['Separate boosting']
print(f'Pooled MAE reduction relative to separate boosting: {improvement:.2f}% (negative means worse)')
print('Best approach on this held-out period:',maes.idxmin())
p.to_csv(OUT/'test_predictions.csv',index=False)
joblib.dump({'models':models,'features':features,'lag_months':LAG_MONTHS,
             'selected_configs':selections},OUT/'fitted_models.joblib')


In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
metrics.set_index('model').MAE_minutes.plot.barh(ax=axes[0],color=['#888888','#bbbbbb','#527d9c','#264d34'])
axes[0].set_xlabel('Mean absolute error (minutes)'); axes[0].set_title('Held-out prediction error')
by_route.pivot(index='group',columns='model',values='MAE_minutes')[names].plot.bar(ax=axes[1])
axes[1].set_ylabel('MAE (minutes)'); axes[1].set_xlabel('Route'); axes[1].set_title('Errors by route')
axes[1].legend(fontsize=8)
axes[1].set_ylim(0,1.15)
fig.tight_layout(); fig.savefig(OUT/'baseline_comparison.png',dpi=160); plt.show()
fig, ax = plt.subplots(figsize=(9,4))
by_month.pivot(index='group',columns='model',values='MAE_minutes')[names].plot(marker='o',ax=ax)
ax.set_ylabel('MAE (minutes)'); ax.set_xlabel('Test month'); ax.set_title('Performance over held-out months')
fig.tight_layout(); fig.savefig(OUT/'monthly_errors.png',dpi=160); plt.show()


## 5. Save results and interpret them
The ZIP contains cleaned and quarantined rows, coverage, modeling features, validation scores, final predictions, metrics, charts, model objects and run metadata. Keep the raw CSV separately. Model objects should only be loaded from trusted sources.

**What you can conclude:** whether pooling reduces aggregate prediction error for these three routes, on this cleaned sample and held-out period. **What you cannot conclude:** causality, delay propagation, superiority on individual bus arrivals, or that shared roads caused any benefit. More routes and matched-size controls would be needed for the connectivity claim.

Next research extension: repeat the comparison with reduced target-route training history, keeping the future test rows fixed. First investigate the quarantined variant groups with the source documentation; do not assume their trip counts represent disjoint observations.

Suggested AI-use disclosure (adapt to your course): “I used ChatGPT/Codex to help construct and check the data-cleaning and baseline-modeling notebook. I reviewed its preprocessing, chronological splits and reported results.”


In [ ]:
metadata = {'audit':audit,'source_sha256':hashlib.sha256(Path(CSV_PATH).read_bytes()).hexdigest(),
    'lag_months':LAG_MONTHS,'selected_configurations':selections,'configurations':configs,
    'features':features,'test_months':[str(x) for x in sorted(test.period.unique())],
    'refit_last_month':str(refit.period.max()),'sklearn_version':sklearn.__version__,
    'pandas_version':pd.__version__, 'pooled_MAE_reduction_percent':float(improvement),
    'publication_timing':'Retrospective two-month availability assumption; actual release dates unverified.'}
(OUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2))
summary = (f"Source rows: {len(raw):,}; cleaned rows: {len(d):,}.\n"
           f"Ambiguous rows quarantined: {audit['ambiguous_rows_quarantined']:,}.\n"
           f"Test period: {test.period.min()} to {test.period.max()}; {len(test):,} rows.\n\n"
           +metrics.to_string(index=False)+f"\n\nPooled reduction vs separate: {improvement:.2f}%.\n"
           +"Descriptive retrospective results; no causal claim. Actual publication timing unverified.\n")
(OUT/'RESULTS.txt').write_text(summary)
archive = shutil.make_archive('bus_baseline_results','zip',OUT)
print(summary)
print('Saved:',archive)
# Set True to download outputs automatically in Colab after running.
DOWNLOAD_RESULTS = False
if DOWNLOAD_RESULTS:
    from google.colab import files
    files.download(archive)
